In [1]:
# DataDoctor3b2a compatible Python 3.11 - version complète
# Inclut: profil interactif, encodage catégoriel, sauvegarde, et traitement complet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import re
from datetime import datetime
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from scipy import stats
from tqdm.notebook import tqdm
from IPython.display import display, HTML
from tabulate import tabulate

# Suppression des avertissements
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
#plt.style.use('seaborn')
plt.style.use('seaborn-v0_8-darkgrid')

# Modules facultatifs
try:
    import missingno as msno
except ImportError:
    msno = None

try:
    import plotly.express as px
except ImportError:
    px = None

try:
    from category_encoders import TargetEncoder
except ImportError:
    TargetEncoder = None

try:
    from ydata_profiling import ProfileReport
except ImportError:
    ProfileReport = None


class DataDoctorConfig:
    def __init__(self):
        self.source_file = "data.csv"
        self.output_format = "csv"
        self.detect_encoding = True
        self.verbose = True
        self.progress_bar = True
        self.max_missing_percent = 30
        self.impute_method = 'auto'
        self.outlier_method = 'iqr'
        self.interactive = True
        self.generate_profile = True
        self.convert_dtypes = True
        self.normalize_text = True
        self.text_case = 'title'
        self.max_categories = 20
        self.remove_duplicates = True


class DataDoctor:
    def __init__(self, config: DataDoctorConfig):
        self.config = config
        self.df = None
        self.df_clean = None
        self.log_entries = []

    def log(self, msg):
        if self.config.verbose:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")
        self.log_entries.append(msg)

    def load_data(self):
        encodings = ['utf-8', 'latin1', 'ISO-8859-1', 'cp1252'] if self.config.detect_encoding else [None]
        for enc in encodings:
            try:
                self.df = pd.read_csv(self.config.source_file, encoding=enc, low_memory=False)
                self.df_clean = self.df.copy()
                self.log(f"Chargement réussi avec encodage: {enc or 'default'}")
                return
            except Exception:
                continue
        raise ValueError("Échec du chargement du fichier.")

    def remove_empty_and_constant_columns(self):
        self.log("Début de la suppression des colonnes vides et constantes")
        na_cols = self.df_clean.columns[self.df_clean.isna().all()].tolist()
        self.df_clean.dropna(axis=1, how='all', inplace=True)
        constant_cols = [col for col in self.df_clean.columns if self.df_clean[col].nunique() <= 1]
        self.df_clean.drop(columns=constant_cols, inplace=True)
        self.log(f"Colonnes supprimées (vides + constantes): {na_cols + constant_cols}")

    def remove_duplicates(self):
        self.log("Début de la suppression des doublons")
        if self.config.remove_duplicates:
            before = len(self.df_clean)
            self.df_clean.drop_duplicates(inplace=True)
            after = len(self.df_clean)
            self.log(f"Doublons supprimés: {before - after}")

    def clean_missing(self):
        self.log("Début du traitement des colonnes à forte proportion de valeurs manquantes")
        na_pct = self.df_clean.isna().mean() * 100
        to_drop = na_pct[na_pct > self.config.max_missing_percent].index.tolist()
        self.df_clean.drop(columns=to_drop, inplace=True)
        self.log(f"Colonnes supprimées (> {self.config.max_missing_percent}% NA): {to_drop}")

    def impute(self):
        self.log("Début de l'imputation des valeurs manquantes")
        num_cols = self.df_clean.select_dtypes(include=np.number).columns
        for col in num_cols:
            if self.df_clean[col].isna().any():
                method = 'mean' if self.df_clean[col].skew() < 1 else 'median'
                val = getattr(self.df_clean[col], method)()
                self.df_clean[col].fillna(val, inplace=True)
                self.log(f"Imputation {method} sur {col}: {val:.2f}")

    def fix_outliers(self):
        self.log("Début de la détection et correction des outliers")
        if self.config.outlier_method != 'iqr':
            return
        for col in self.df_clean.select_dtypes(include=np.number):
            q1 = self.df_clean[col].quantile(0.25)
            q3 = self.df_clean[col].quantile(0.75)
            iqr = q3 - q1
            low = q1 - 1.5 * iqr
            high = q3 + 1.5 * iqr
            before = self.df_clean[col].copy()
            self.df_clean[col] = np.clip(before, low, high)
            if not np.array_equal(before, self.df_clean[col]):
                self.log(f"Outliers corrigés: {col}")

    def normalize_text(self):
        if not self.config.normalize_text:
            return
        for col in self.df_clean.select_dtypes(include='object'):
            self.df_clean[col] = self.df_clean[col].astype(str).str.strip()
            if self.config.text_case == 'lower':
                self.df_clean[col] = self.df_clean[col].str.lower()
            elif self.config.text_case == 'upper':
                self.df_clean[col] = self.df_clean[col].str.upper()
            elif self.config.text_case == 'title':
                self.df_clean[col] = self.df_clean[col].str.title()
            self.log(f"Colonne normalisée: {col}")

    def encode_categoricals(self):
        if TargetEncoder is None:
            self.log("Encodage catégoriel ignoré (TargetEncoder absent)")
            return
        for col in self.df_clean.select_dtypes(include='object'):
            if self.df_clean[col].nunique() > self.config.max_categories:
                encoder = TargetEncoder()
                self.df_clean[col] = encoder.fit_transform(self.df_clean[col], self.df_clean[col])
                self.log(f"Encodage TargetEncoder appliqué: {col}")

    def convert_dtypes(self):
        if self.config.convert_dtypes:
            self.df_clean = self.df_clean.convert_dtypes()
            self.log("Types de données optimisés")

    def profile(self):
        if self.config.generate_profile and ProfileReport:
            profile = ProfileReport(self.df_clean, title="Profil DataDoctor")
            profile.to_file("rapport.html")
            self.log("Rapport sauvegardé dans rapport.html")

    def visualize_missing(self):
        if self.config.interactive and msno:
            msno.matrix(self.df_clean)
            plt.title("Valeurs manquantes")
            plt.show()

    def save(self):
        out = self.config.source_file.replace('.csv', '-doctored.csv')
        self.df_clean.to_csv(out, index=False)
        self.log(f"Fichier sauvegardé: {out}")

    def run(self):
        self.load_data()
        self.remove_empty_and_constant_columns()
        self.remove_duplicates()
        self.clean_missing()
        self.impute()
        self.fix_outliers()
        self.normalize_text()
        self.encode_categoricals()
        self.convert_dtypes()
        self.visualize_missing()
        self.profile()
        self.save()
        return self.df_clean


if __name__ == '__main__':
    cfg = DataDoctorConfig()
    cfg.source_file = "dataset_v2.csv"  # Fichier source
    doc = DataDoctor(cfg)
    df_final = doc.run()
    display(df_final.head())


[12:55:12] Chargement réussi avec encodage: utf-8
[12:55:12] Début de la suppression des colonnes vides et constantes
[12:55:12] Colonnes supprimées (vides + constantes): ['shipping_cost']
[12:55:12] Début de la suppression des doublons
[12:55:12] Doublons supprimés: 0
[12:55:12] Début du traitement des colonnes à forte proportion de valeurs manquantes
[12:55:12] Colonnes supprimées (> 30% NA): []
[12:55:12] Début de l'imputation des valeurs manquantes
[12:55:12] Imputation mean sur age: 35.50
[12:55:12] Imputation mean sur income: 11.68
[12:55:12] Début de la détection et correction des outliers
[12:55:12] Outliers corrigés: age
[12:55:12] Outliers corrigés: income
[12:55:12] Outliers corrigés: experience_years
[12:55:12] Outliers corrigés: nb_purchases
[12:55:12] Outliers corrigés: discount
[12:55:12] Outliers corrigés: purchase_amount
[12:55:12] Colonne normalisée: city
[12:55:12] Colonne normalisée: signup_date
[12:55:12] Encodage catégoriel ignoré (TargetEncoder absent)
[12:55:12]

,age,income,experience_years,city,is_customer,nb_purchases,signup_date,discount,purchase_amount
0,46.734271,52.824389,0.155191,Lyon,0,5,2024-03-07 00:48:06,251.4807,1120.741627
1,48.881277,<NA>,1.555864,Toulouse,1,3,2024-05-28 05:13:40,244.854535,1136.39861
2,32.987377,22.721589,2.86092,Paris,0,7,2025-01-15 04:25:04,170.555118,923.973969
3,39.7797,24.419138,2.218682,Lyon,0,0,2021-05-15 23:07:22,209.754576,1045.155189
4,46.570973,22.007126,2.64339,Lyon,0,3,2023-09-21 15:29:42,239.79592,1088.69608
